#  Étape 2 — Preprocessing
**Objectif** : Nettoyer les textes, filtrer les langues non-françaises, et créer les splits train/val/test stratifiés.

In [2]:
# ═══════════════════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Projet_Sentiment_Analysis'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
FIG_DIR     = os.path.join(PROJECT_DIR, 'figures')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models')

!pip install -q langdetect

import pandas as pd
import numpy as np
import re
from langdetect import detect, DetectorFactory
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Reproductibilité
DetectorFactory.seed = 42
np.random.seed(42)

print("✅ Setup terminé")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Setup terminé


In [3]:
# ═══════════════════════════════════════════════════════
# CHARGEMENT
# ═══════════════════════════════════════════════════════
df = pd.read_csv(os.path.join(DATA_DIR, 'dataset_avis_apps.csv'))

COL_TEXTE     = 'avis'
COL_SENTIMENT = 'sentiment'
COL_NOTE      = 'note'
COL_APP       = 'application'

print(f" Dataset brut : {len(df):,} lignes")
print(f" Colonnes : {list(df.columns)}")
df.head()

 Dataset brut : 29,405 lignes
 Colonnes : ['application', 'categorie', 'note', 'sentiment', 'avis', 'date_avis', 'likes']


,application,categorie,note,sentiment,avis,date_avis,likes
0,Attijari Mobile,Banque,1,negative,Depuis quelque mois je n'ai plus accès a l'app...,2026-06-04 14:29:16,0
1,Attijari Mobile,Banque,1,negative,توجد به أخطاء بعد التحدئثات الاخيرة,2026-06-04 01:49:11,0
2,Attijari Mobile,Banque,1,negative,location,2026-06-03 19:42:33,0
3,Attijari Mobile,Banque,1,negative,prob changer l appareil malgré qu il fonctionn...,2026-06-01 15:35:31,0
4,Attijari Mobile,Banque,1,negative,depuis la dernière mise à jour je n'arrive plu...,2026-06-01 08:22:20,0


## 1. Suppression des doublons et valeurs manquantes

In [4]:
# ═══════════════════════════════════════════════════════
# DOUBLONS & VALEURS MANQUANTES
# ═══════════════════════════════════════════════════════
COL_TEXTE = 'avis'
n_avant = len(df)

# Supprimer les lignes avec texte manquant
df = df.dropna(subset=[COL_TEXTE])
df[COL_TEXTE] = df[COL_TEXTE].astype(str)

# Supprimer les doublons exacts (même texte + même app)
df = df.drop_duplicates(subset=[COL_TEXTE, COL_APP], keep='first')

n_apres = len(df)
print(f" Avant : {n_avant:,} → Après : {n_apres:,}")
print(f"   Supprimés : {n_avant - n_apres:,} ({100*(n_avant-n_apres)/n_avant:.1f}%)")

 Avant : 29,405 → Après : 20,335
   Supprimés : 9,070 (30.8%)


## 2. Nettoyage du texte

In [5]:
# ═══════════════════════════════════════════════════════
# FONCTIONS DE NETTOYAGE
# ═══════════════════════════════════════════════════════

def nettoyer_texte(texte):
    """
    Pipeline de nettoyage pour un avis.
    PAS de stemming ni lemmatisation (CamemBERT préfère les formes complètes).
    """
    if not isinstance(texte, str) or len(texte.strip()) == 0:
        return ""

    # 1. Minuscules
    texte = texte.lower()

    # 2. Supprimer les URLs
    texte = re.sub(r'http\S+|www\.\S+', '', texte)

    # 3. Supprimer les emails
    texte = re.sub(r'\S+@\S+', '', texte)

    # 4. Supprimer les emojis et caractères spéciaux Unicode
    texte = re.sub(r'[^\w\s\'\-àâäéèêëïîôùûüÿçœæ.,!?]', ' ', texte)

    # 5. Supprimer les chiffres isolés (garder ceux dans les mots)
    texte = re.sub(r'\b\d+\b', '', texte)

    # 6. Supprimer la ponctuation excessive (garder une seule occurrence)
    texte = re.sub(r'([!?.]){2,}', r'\1', texte)

    # 7. Supprimer les espaces multiples
    texte = re.sub(r'\s+', ' ', texte).strip()

    return texte


# Appliquer le nettoyage
df['texte_clean'] = df[COL_TEXTE].apply(nettoyer_texte)

# Exemples avant/après
print(" Exemples avant/après nettoyage :\n")
for i in range(5):
    idx = df.index[i]
    print(f"  AVANT : {df.loc[idx, COL_TEXTE][:100]}")
    print(f"  APRÈS : {df.loc[idx, 'texte_clean'][:100]}")
    print(f"  {'—'*60}")

 Exemples avant/après nettoyage :

  AVANT : Depuis quelque mois je n'ai plus accès a l'appli depuis mon tel que j'utilise depuis plusieurs année
  APRÈS : depuis quelque mois je n'ai plus accès a l'appli depuis mon tel que j'utilise depuis plusieurs année
  ————————————————————————————————————————————————————————————
  AVANT : توجد به أخطاء بعد التحدئثات الاخيرة
  APRÈS : توجد به أخطاء بعد التحدئثات الاخيرة
  ————————————————————————————————————————————————————————————
  AVANT : location
  APRÈS : location
  ————————————————————————————————————————————————————————————
  AVANT : prob changer l appareil malgré qu il fonctionne avant dans même appareil
  APRÈS : prob changer l appareil malgré qu il fonctionne avant dans même appareil
  ————————————————————————————————————————————————————————————
  AVANT : depuis la dernière mise à jour je n'arrive plus à me connecter. après la saisie du mot de passe, l é
  APRÈS : depuis la dernière mise à jour je n'arrive plus à me connecter. après la s

## 3. Détection de langue — garder le français uniquement

In [6]:
# ═══════════════════════════════════════════════════════
# DÉTECTION DE LANGUE
# ═══════════════════════════════════════════════════════
from langdetect import detect, LangDetectException

def detecter_langue(texte):
    """
    Détecte la langue du texte.
    Retourne 'fr', 'ar', 'en', etc. ou 'unknown' si échec.
    """
    try:
        if len(texte.strip()) < 3:
            return 'unknown'
        return detect(texte)
    except LangDetectException:
        return 'unknown'

print("🔍 Détection de langue en cours... (peut prendre 2-5 min)")
df['langue'] = df['texte_clean'].apply(detecter_langue)

# Distribution des langues
print("\n Distribution des langues détectées :")
lang_counts = df['langue'].value_counts()
for lang, count in lang_counts.head(10).items():
    print(f"   {lang:5s} : {count:>6,} ({100*count/len(df):5.1f}%)")

# Garder seulement le français
n_avant = len(df)
df = df[df['langue'] == 'fr'].copy()
n_apres = len(df)

print(f"\n Filtre français appliqué : {n_avant:,} → {n_apres:,}")
print(f"   Supprimés : {n_avant - n_apres:,} ({100*(n_avant-n_apres)/n_avant:.1f}%)")

🔍 Détection de langue en cours... (peut prendre 2-5 min)

 Distribution des langues détectées :
   fr    : 11,607 ( 57.1%)
   ar    :  2,742 ( 13.5%)
   en    :  1,076 (  5.3%)
   unknown :    709 (  3.5%)
   it    :    649 (  3.2%)
   ca    :    519 (  2.6%)
   ro    :    359 (  1.8%)
   so    :    332 (  1.6%)
   es    :    191 (  0.9%)
   af    :    164 (  0.8%)

 Filtre français appliqué : 20,335 → 11,607
   Supprimés : 8,728 (42.9%)


In [7]:
# ═══════════════════════════════════════════════════════
# FILTRER LES TEXTES TROP COURTS
# ═══════════════════════════════════════════════════════
df['nb_mots_clean'] = df['texte_clean'].apply(lambda x: len(x.split()))

n_avant = len(df)
df = df[df['nb_mots_clean'] >= 3].copy()  # Minimum 3 mots
n_apres = len(df)

print(f"🔍 Filtre longueur (≥3 mots) : {n_avant:,} → {n_apres:,}")
print(f"   Supprimés : {n_avant - n_apres:,}")

🔍 Filtre longueur (≥3 mots) : 11,607 → 9,984
   Supprimés : 1,623


## 4. Encodage des labels


In [8]:
# ═══════════════════════════════════════════════════════
# ENCODAGE DES LABELS
# ═══════════════════════════════════════════════════════

LABEL_MAP = {
    'négatif': 0, 'negatif': 0, 'negative': 0,
    'neutre': 1, 'neutral': 1,
    'positif': 2, 'positive': 2
}

# Mapping inverse pour l'affichage
LABEL_NAMES = {0: 'négatif', 1: 'neutre', 2: 'positif'}

df['label'] = df[COL_SENTIMENT].map(LABEL_MAP)

# Vérifier qu'il n'y a pas de NaN (= labels non mappés)
n_unmapped = df['label'].isna().sum()
if n_unmapped > 0:
    print(f" {n_unmapped} labels non reconnus :")
    print(df[df['label'].isna()][COL_SENTIMENT].unique())
else:
    print(" Tous les labels sont mappés correctement")

df['label'] = df['label'].astype(int)

print(f"\n Distribution finale des labels :")
for label, name in LABEL_NAMES.items():
    n = (df['label'] == label).sum()
    print(f"   {name:10s} (label={label}) : {n:>6,} ({100*n/len(df):5.1f}%)")

 Tous les labels sont mappés correctement

 Distribution finale des labels :
   négatif    (label=0) :  3,737 ( 37.4%)
   neutre     (label=1) :    754 (  7.6%)
   positif    (label=2) :  5,493 ( 55.0%)


## 5. Split train / val / test (80/10/10 stratifié)

In [9]:
# ═══════════════════════════════════════════════════════
# SPLIT STRATIFIÉ 80/10/10
# ═══════════════════════════════════════════════════════

# Premier split : 80% train, 20% temp
df_train, df_temp = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

# Deuxième split : 50/50 du temp → 10% val, 10% test
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, random_state=42, stratify=df_temp['label']
)

print(f" Splits :")
print(f"   Train : {len(df_train):>6,} ({100*len(df_train)/len(df):.1f}%)")
print(f"   Val   : {len(df_val):>6,} ({100*len(df_val)/len(df):.1f}%)")
print(f"   Test  : {len(df_test):>6,} ({100*len(df_test)/len(df):.1f}%)")

# Vérifier la stratification
print(f"\n Vérification de la stratification (% de chaque classe) :")
for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    dist = split_df['label'].value_counts(normalize=True).sort_index()
    print(f"   {name:5s} : ", end="")
    for label in sorted(dist.index):
        print(f"{LABEL_NAMES[label]}={100*dist[label]:.1f}%  ", end="")
    print()

 Splits :
   Train :  7,987 (80.0%)
   Val   :    998 (10.0%)
   Test  :    999 (10.0%)

 Vérification de la stratification (% de chaque classe) :
   Train : négatif=37.4%  neutre=7.5%  positif=55.0%  
   Val   : négatif=37.4%  neutre=7.6%  positif=55.0%  
   Test  : négatif=37.4%  neutre=7.5%  positif=55.1%  


In [10]:
# ═══════════════════════════════════════════════════════
# SAUVEGARDE DES SPLITS
# ═══════════════════════════════════════════════════════

# Colonnes à garder dans les fichiers finaux
cols_to_save = [COL_TEXTE, 'texte_clean', COL_SENTIMENT, 'label', COL_APP, COL_NOTE]
cols_to_save = [c for c in cols_to_save if c in df.columns]  # garder seulement les existantes

df_train[cols_to_save].to_csv(os.path.join(DATA_DIR, 'train.csv'), index=False)
df_val[cols_to_save].to_csv(os.path.join(DATA_DIR, 'val.csv'), index=False)
df_test[cols_to_save].to_csv(os.path.join(DATA_DIR, 'test.csv'), index=False)

print(" Fichiers sauvegardés :")
print(f"    {os.path.join(DATA_DIR, 'train.csv')}")
print(f"    {os.path.join(DATA_DIR, 'val.csv')}")
print(f"    {os.path.join(DATA_DIR, 'test.csv')}")

 Fichiers sauvegardés :
    /content/drive/MyDrive/Projet_Sentiment_Analysis/data/train.csv
    /content/drive/MyDrive/Projet_Sentiment_Analysis/data/val.csv
    /content/drive/MyDrive/Projet_Sentiment_Analysis/data/test.csv


In [11]:
# ═══════════════════════════════════════════════════════
# CALCUL DES CLASS WEIGHTS (à réutiliser)
# ═══════════════════════════════════════════════════════
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df_train['label'].values
)

print(" Class weights calculés (pour compenser le déséquilibre) :")
for label, weight in enumerate(class_weights):
    print(f"   {LABEL_NAMES[label]:10s} (label={label}) : weight = {weight:.4f}")

print(f"\n À utiliser dans PyTorch :")
print(f"   class_weights = torch.tensor({list(np.round(class_weights, 4))}, dtype=torch.float)")

 Class weights calculés (pour compenser le déséquilibre) :
   négatif    (label=0) : weight = 0.8904
   neutre     (label=1) : weight = 4.4151
   positif    (label=2) : weight = 0.6059

 À utiliser dans PyTorch :
   class_weights = torch.tensor([np.float64(0.8904), np.float64(4.4151), np.float64(0.6059)], dtype=torch.float)


In [12]:
# ═══════════════════════════════════════════════════════
# RÉSUMÉ DU PREPROCESSING
# ═══════════════════════════════════════════════════════
print("="*60)
print(" RÉSUMÉ DU PREPROCESSING")
print("="*60)
print(f"""
 Dataset initial          : 29 405 avis
   Après déduplication      : {n_apres:,}
   Après filtre français    : {len(df):,}
   Après filtre longueur    : {len(df):,}

 Splits sauvegardés :
   Train : {len(df_train):,} avis
   Val   : {len(df_val):,} avis
   Test  : {len(df_test):,} avis

 Transformations appliquées :
   ✅ Minuscules
   ✅ Suppression URLs, emails, emojis
   ✅ Suppression chiffres isolés
   ✅ Normalisation ponctuation
   ✅ Filtrage langdetect (français uniquement)
   ✅ Filtrage textes < 3 mots
   ❌ PAS de stemming (CamemBERT)
   ❌ PAS de lemmatisation (CamemBERT)

✅ Preprocessing terminé !
""")

 RÉSUMÉ DU PREPROCESSING

 Dataset initial          : 29 405 avis
   Après déduplication      : 9,984
   Après filtre français    : 9,984
   Après filtre longueur    : 9,984

 Splits sauvegardés :
   Train : 7,987 avis
   Val   : 998 avis
   Test  : 999 avis

 Transformations appliquées :
   ✅ Minuscules
   ✅ Suppression URLs, emails, emojis
   ✅ Suppression chiffres isolés
   ✅ Normalisation ponctuation
   ✅ Filtrage langdetect (français uniquement)
   ✅ Filtrage textes < 3 mots
   ❌ PAS de stemming (CamemBERT)
   ❌ PAS de lemmatisation (CamemBERT)

✅ Preprocessing terminé !

